# Session 1 — Introduction, CMS Data, and Coffea Basics

This is the first part of the **CMS DAS bbMET analysis exercise**. In this session we connect the physics motivation from the CMS search for dark matter produced with bottom quarks (bb + MET) to the concrete data format (NanoAOD) and tools (Coffea) that we will use in the following sessions.

**Learning objectives:**
- Understand dark matter collider searches, with a focus on **bb + MET** final states
- Understand CMS data formats and NanoAOD, and how they encode jets, b-jets, leptons, and MET
- Load data using Coffea and explore event content in a bbMET-style analysis
- Produce basic kinematic plots (jets, leptons, MET) that appear in the full analysis


> For context, many of the concepts here are used in the CMS bb + MET analysis documented in JHEP 02 (2025) 050. We only implement a simplified version suitable for a hands-on exercise.

![Placeholder: overview of bb + MET search strategy](figures/session1_fig_bbmet_overview.png)

---
## 1. Introduction to Dark Matter

Cosmological measurements (e.g. Planck) tell us that about **26%** of the energy density of the universe is in the form of **dark matter (DM)**. Evidence comes from galaxy rotation curves, gravitational lensing, and the cosmic microwave background, but we do not yet know the particle nature of DM.

A broad class of models assumes **weakly interacting massive particles (WIMPs)** with masses of a few GeV–TeV. If such particles couple to quarks or gluons, they can be **produced at the LHC**. Because they are neutral and weakly interacting, they traverse the detector, leading to an apparent imbalance in the transverse momentum of visible particles.

At colliders like the LHC we therefore search for DM by looking for **large missing transverse energy (MET)**, often written as \(p_T^{\text{miss}}\), recoiling against visible objects such as jets or leptons. In this CMS DAS exercise we focus on the case where the DM is produced together with a **pair of bottom quarks (bb)**, leading to a **bb + MET** final state.

### Why colliders?

- We can produce DM in the lab if it couples to quarks, gluons, or electroweak particles.
- The experimental signature is **MET** (imbalance in transverse momentum) when DM particles leave the detector.
- Heavy-flavor jets (b-jets) are important in many models (e.g. the **2HDM+a** benchmark used in the CMS bb + MET analysis) where the mediator couples preferentially to the Higgs sector and to bottom quarks.
- bb + MET searches are **complementary** to other X + MET channels (monojet, mono-Z, tt̄ + MET, etc.), probing regions of parameter space with large couplings to b quarks and high values of \(\tan\beta\).

In this DAS exercise we will not implement the full model, but we will reproduce the main **analysis ingredients** used in such a search: defining physics objects, selecting a signal-like region, and inspecting key distributions such as MET.

### Signal process (simplified)

We consider a simplified process inspired by the **2HDM+a** model used in the CMS analysis:

$$pp \to b\bar{b} + \chi\bar{\chi}$$

where \(\chi\) is the dark matter particle (invisible). In 2HDM+a, the interaction proceeds via additional scalar and pseudoscalar bosons; their couplings to bottom quarks are enhanced at large \(\tan\beta\), making **bb + MET** a powerful channel.

**Experimental signature:**
- Two or more **b-jets**, originating from bottom quarks
- Large **MET**, carried away by the DM particles
- **No isolated leptons**, to suppress W+jets and semileptonic tt̄ backgrounds

Later in the exercise you will build selections that move from a loose bb + jets sample towards a **signal region** resembling that used in the full analysis.

![Placeholder: 2HDM+a bb + DM Feynman diagrams](figures/session1_fig_feynman_bbmet.png)

---
## 2. Dark Matter Searches at Colliders

Collider DM searches are often summarised as **X + MET** signatures, where X is one or more visible objects (jets, photons, W/Z bosons, top quarks, …) and MET is carried by invisible particles (DM or neutrinos).

In the **bb + MET** channel:

| Observable | Role |
|-----------|------|
| **MET** | Indicates invisible particles carrying away momentum (DM + neutrinos) |
| **b-jets** | Tag production in association with bottom quarks; enhanced in high-\(\tan\beta\) 2HDM+a scenarios |
| **Lepton veto** | Reduces W+jets and semileptonic tt̄, where a charged lepton is present |

**Main backgrounds in a realistic bbMET analysis (as in JHEP 02 (2025) 050):**
- **Z(→νν) + jets**, especially with b-jets: genuine MET from neutrinos, dominant at high MET.
- **tt̄** and **single-top**: b-jets + MET from W→ℓν; reduced by lepton veto but still important.
- **W + jets**: MET from W→ℓν; strongly reduced by the lepton veto.
- **QCD multijet**: fake MET from mismeasured jets; suppressed by MET cuts and angular requirements.

In this DAS exercise we focus on the **object-level ingredients and simple selections** that these full analyses rely on; full background estimation and fits come in later sessions.

![Placeholder: schematic comparison of signal and background MET shapes](figures/session1_fig_met_shapes.png)

---
## 3. Overview of the CMS Detector

CMS is a general-purpose detector at the LHC built around a **superconducting solenoid** (3.8 T) with:
- **Silicon tracker** — measures charged-particle trajectories and momenta.
- **Electromagnetic calorimeter (ECAL)** — measures energies of electrons and photons.
- **Hadron calorimeter (HCAL)** — measures hadronic energy (jets).
- **Muon system** — identifies and measures muons outside the solenoid.

In modern CMS analyses, events are reconstructed with the **particle-flow (PF)** algorithm, which combines information from all subsystems into a list of PF candidates (charged hadrons, neutral hadrons, photons, electrons, muons). From these, we build:
- **Jets** (clustered PF candidates)
- **b-tagged jets** (jets tagged as likely containing a b hadron)
- **Leptons** (electrons, muons, taus)
- **MET** (negative vector sum of PF candidates in the transverse plane)

Neutrinos (and DM) do not leave direct signals, so an imbalance in the PF candidates’ transverse momentum defines **MET**, our main handle on invisible particles.

![Placeholder: CMS detector cartoon](figures/session1_fig_cms_detector.png)

*Discussion:* Why can’t we "see" dark matter in the detector, and why is MET only defined in the transverse plane (not along the beam)?

---
## 4. CMS Data Formats

CMS stores collision data and simulation in **ROOT** files. Processing levels:

- **RAW** — raw detector data
- **AOD** — analysis object data (reconstructed objects)
- **NanoAOD** — reduced format with only the most used branches, designed for analysis

We will use **NanoAOD** because it is small and fast to process.

---
## 5. What is NanoAOD?

NanoAOD contains:
- **Jets** (pt, η, φ, mass, b-tag discriminants, ID)
- **MET** (pt, φ)
- **Electrons, Muons** (kinematics, ID, isolation)
- **Triggers, generator info** (for MC)

Branch names follow the pattern `Object_variable`, e.g. `Jet_pt`, `MET_pt`. In Coffea we access them as `events.Jet.pt`, `events.MET.pt`.

---
## 6. Introduction to Coffea

**Coffea** (Columnar Object Framework for Effective Analysis) is a Python library for HEP analysis. It:
- Reads ROOT/NanoAOD with **uproot**
- Uses **Awkward Array** for jagged (variable-length) data
- Supports **columnar** operations (efficient over many events)

Install (if needed):
```bash
pip install coffea matplotlib hist uproot
```

In [ ]:
# Check that Coffea and related packages are available
try:
    import coffea
    import awkward as ak
    import numpy as np
    print("coffea:", coffea.__version__)
    print("awkward:", ak.__version__)
    print("numpy:", np.__version__)
except ImportError as e:
    print("Missing package:", e)

---
## 7. Loading NanoAOD Files

We use `NanoEventsFactory` from Coffea to load NanoAOD. You need a path to a NanoAOD file (local or XRootD).

**For the school:** Your instructor will provide a small sample file path. Replace the path below with yours.

In [ ]:
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema

NanoAODSchema.warn_missing_crossrefs = False


def load_events(filepath):
    """Load one NanoAOD file and return events (Coffea NanoEvents)."""
    return NanoEventsFactory.from_root(
        {filepath: "Events"},
        schemaclass=NanoAODSchema,
        metadata={"dataset": "nanoaod"},
        mode="eager",
    ).events()

# Example: local file (replace with your file path)
filepath = "/eos/cms/store/group/phys_susy/sus-23-008/cmsdas2026/2017/MET-Run2017B-02Apr2020-v1/FA2D6DDA-28C1-8A4E-B170-B7217A9ED411.root"
events = load_events(filepath)
print("Number of events:", len(events))

print("To run this cell, set filepath to a NanoAOD file and uncomment the lines above.")

### Exercise 1.1 — Load and inspect a NanoAOD file

Choose **one NanoAOD file yourself** (from the paths provided by the instructors) and set `filepath` accordingly. Use the `load_events(filepath)` helper defined above to create an `events` object.

Once `events` is loaded, print:
- **The total number of events**, and
- **The run, luminosity block, and event numbers for a few events** (e.g. first 5) using `events.run`, `events.luminosityBlock`, and `events.event`.

If you don’t have a real file yet, you can skip the execution or use a dummy path and catch the error.

### Option: use the 2017 config to pick files automatically

If the 2017 samples are available under the configured path defined in `config/datasets_2017.py`, you can load **one data file and one background file** using that config. The next cell:
- Adds the project root (with the `config` directory) to `sys.path`,
- Imports `get_one_file_per_group` from `config/datasets_2017.py`, and
- Uses it to define `events` from a data file (and `events_bkg` from a background file).

This is the **recommended way** to load a small sample for the school; you can then solve Exercise 1.1 using this `events` object.

In [ ]:
# Exercise 1.1 — choose a file, load it, and inspect basic quantities.

# 1) Set your chosen NanoAOD file path here, then load events:
# filepath = "..."  # e.g. a path provided by the instructors
# events = load_events(filepath)

# 2) Uncomment the lines below to inspect the sample:
# print("Total number of events:", len(events))
# print("run (first 5):", events.run[:5])
# print("luminosityBlock (first 5):", events.luminosityBlock[:5])
# print("event (first 5):", events.event[:5])

---
## 8. Inspecting Event Content

Once events are loaded, inspect the available collections. In Coffea, `events` is an Awkward array of events; each attribute (e.g. `Jet`, `MET`) is a jagged structure.

In [ ]:
# Uncomment after loading events
# print("Collections:", [k for k in dir(events) if not k.startswith("_")])
# print("\nJets: ", events.Jet)
# print("MET: ", events.MET.pt[:5], "... (first 5 events)")
# print("Number of jets per event (first 10):", ak.num(events.Jet)[:10])

### Key branches to explore

| Collection | Useful branches |
|------------|------------------|
| `events.Jet` | `pt`, `eta`, `phi`, `mass`, `jetId`, `btagDeepFlavB` |
| `events.MET` | `pt`, `phi` |
| `events.Electron` | `pt`, `eta`, `cutBased` |
| `events.Muon` | `pt`, `eta`, `tightId`, `pfRelIso04_all` |

In [ ]:
# Exercise 1.2: Print the list of Jet branch names (field names)
# Hint: events.Jet.fields or dir(events.Jet)
# Your code:

---
## 9. Basic Plotting

We use **matplotlib** (or **hist**) to plot distributions. With Awkward Arrays, we often **flatten** per-object quantities (e.g. jet pT) to get one entry per object, or use per-event quantities (e.g. MET) directly.

In [ ]:
import matplotlib.pyplot as plt
import mplhep as hep

hep.style.use("CMS")

# Example: plot MET for the first 10k events (if available)
# met = events.MET.pt
# plt.figure(figsize=(6,4))
# plt.hist(ak.to_numpy(met), bins=50, range=(0, 400), edgecolor="black", alpha=0.7)
# plt.xlabel("MET [GeV]")
# plt.ylabel("Events")
# plt.title("Missing transverse energy")
# plt.show()

print("Uncomment and run after loading events.")

### Exercise 1.3 — Jet pT
Fill a histogram of **jet pT** for all jets in your sample. Use `events.Jet.pt`, flatten with `ak.flatten(events.Jet.pt)`, and plot with matplotlib. Use bins from 0 to 500 GeV.

In [ ]:
# Exercise 1.3 — Jet pT
# Example: compute flattened jet pt
jpt = ak.flatten(events.Jet.pt)
# TODO: add a histogram of jpt with matplotlib.

### Exercise 1.4 — Jet multiplicity
Plot the **number of jets per event** using `ak.num(events.Jet)`. Use bins from 0 to 15. Label axes and add a title.

In [ ]:
# Exercise 1.4 — Jet multiplicity
# Example: compute number of jets per event
njets = ak.num(events.Jet)
# TODO: make a histogram of njets.

### Exercise 1.5 — Lepton content
Inspect basic kinematics for electrons and muons using the loaded `events` object:
- Print the first few values of **pt, eta, and phi** for `events.Electron` and `events.Muon`.
- Count how many events have at least one electron or at least one muon.

In [ ]:
# Exercise 1.5 — Lepton content
# Example: inspect basic kinematics for electrons
print("Electron pt (first 5 events):", events.Electron.pt[:5])
# TODO: add prints for eta/phi and for muons, and counts of events with >=1 lepton.

### Exercise 1.6 — Lepton kinematic plots
Make histograms of **pt**, **eta**, and **phi** for electrons and muons:
- Use `ak.flatten(events.Electron.pt)` and `ak.flatten(events.Muon.pt)` for pt,
- Similarly flatten `eta` and `phi`.

Plot at least one distribution for electrons and one for muons.

In [ ]:
# Exercise 1.6 — Lepton kinematic plots
# Example: make one pt histogram for electrons
el_pt = ak.flatten(events.Electron.pt)
# TODO: add histograms for muon pt and for eta/phi.

---
## 10. Summary — Session 1

- **Dark matter** at colliders is searched for via MET and (in our case) b-jets.
- **NanoAOD** is a reduced ROOT format; we access it with **Coffea** and **Awkward Arrays**.
- **Loading:** `NanoEventsFactory.from_root({filepath: "Events"}, schemaclass=NanoAODSchema).events()`
- **Inspecting:** `events.Jet`, `events.MET`, etc.; use `ak.flatten` for per-object plots and `ak.num` for multiplicities.
- **Plotting:** Use matplotlib (or hist) with `ak.to_numpy()` for histogram inputs.

**Next session:** We will apply jet quality cuts, b-tagging, and lepton veto to define a clean object selection.